# R04 · Model selection and AE-versus-DAE statistics (Tables 9, 10, S3; Section 4.7; Fig. 8 data)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-account>/nids-partition-distortion-ae-dae/blob/main/notebooks/reproduction/R04_Selection_and_Final_Statistics.ipynb)

Recomputes the selection ranking, final-test levels, paired DAE − AE contrasts with exact bootstrap intervals, paired tests, per-fingerprint recall, score geometry, family recall and the development-versus-final comparison. Optionally re-derives the archive summaries from the 36 prediction archives.

**How to run:** Runtime → Run all. The first run in a fresh Colab runtime installs the pinned library versions and restarts the runtime once; then choose Run all again.

<details><summary>Source and conversion notes</summary>

Reproduce model selection and the downstream AE-versus-DAE results (manuscript Section 4.7;
Tables 9 and 10; Table S3; Fig. 8 data).

Default mode uses only files inside the repository:
  artifacts/<dataset>/model_selection/{selected_models_manifest.json, confirmation_seed_results.csv, selection_ranking.csv}
  artifacts/<dataset>/final_evaluation/final_seed_results.csv
  artifacts/<dataset>/statistical_analysis/{paired_seed_values.csv, paired_seed_bootstrap_effects.csv}
  artifacts/<dataset>/*/derived_from_archives/*.csv   (per-run summaries computed from the prediction archives)

``--archives DIR`` (or the environment variable NIDS_PREDICTION_ARCHIVES) additionally re-derives every
``derived_from_archives`` table from the 36 observation-level prediction archives (external, see
external_artifacts/README.md), checks their SHA-256 against external_artifacts/prediction_archive_manifest.csv
and fails if any recomputed value differs from the committed tables.

Outputs: results/tables/Table_3_4_* ... Table_3_10_*, Table_S3_family_recall.csv,
results/figure_source_data/Figure_3_5_data.csv, results/claims_check/reproduce_final_statistics.csv

Mechanical changes made when copying the .py files into this notebook:
`from src... import ...` lines are removed (the library code is defined in the cells above);
`sys.path.insert(...)` and `from __future__ import annotations` are removed;
`Path(__file__)...` repository roots are replaced by `REPO_ROOT` from the setup cell;
the command-line interface of `src/utils/hashing.py` is omitted;
command-line arguments of `reproduce_final_statistics.py` are replaced by the parameter cell;
`sys.exit(L.finish())` becomes `assert L.finish() == 0`.
</details>

In [ ]:
# ===== Setup: run this cell first =====
# REPO_SOURCE = "drive": the repository is in your Google Drive, either as the unzipped folder
#   `nids-partition-distortion-ae-dae` or as the zip file `nids-partition-distortion-ae-dae.zip`, anywhere
#   in My Drive (up to four folders deep). The cell finds it; set DRIVE_REPO only to point at one copy.
# REPO_SOURCE = "github": after the repository is on GitHub (fill in GITHUB_URL).
REPO_SOURCE = "drive"                                                    # "drive" or "github"
DRIVE_REPO = "/content/drive/MyDrive/nids-partition-distortion-ae-dae"   # optional hint
GITHUB_URL = "https://github.com/<your-account>/nids-partition-distortion-ae-dae.git"

import os, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "nids-partition-distortion-ae-dae"


def _is_repo(p: Path) -> bool:
    return (p / "artifacts" / "ARTIFACT_PROVENANCE.csv").is_file()


def _find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if _is_repo(p):
            return p
    raise FileNotFoundError(f"repository folder not found above {start}")


def _locate_in_drive(hint: Path, drive_root: Path) -> Path:
    # 1. the hint itself, a folder above it, or a nested copy below it (unzipping often creates name/name/)
    for c in [hint, *hint.parents, hint / REPO_NAME]:
        if _is_repo(c):
            return c
    # 2. the unzipped folder anywhere in My Drive, up to four levels deep
    for depth in range(0, 4):
        pattern = "/".join(["*"] * depth + ["artifacts", "ARTIFACT_PROVENANCE.csv"])
        hits = sorted(drive_root.glob(pattern))
        if hits:
            if len(hits) > 1:
                print("Several copies found; using the first (set DRIVE_REPO to choose):", *[h.parents[1] for h in hits], sep="\n  ")
            return hits[0].parents[1]
    # 3. the zip file anywhere in My Drive (Google Drive does not unzip uploads): extract it to /content
    for depth in range(0, 4):
        zips = sorted(drive_root.glob("/".join(["*"] * depth + [REPO_NAME + "*.zip"])))
        if zips:
            dest = Path("/content/repo_from_zip")
            print(f"Found {zips[0]} - extracting to {dest} (results will be written there, not to Drive)")
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(dest)
            return _locate_in_drive(dest, dest)
    near = [p.name for p in drive_root.iterdir() if "nids" in p.name.lower() or "partition" in p.name.lower()] if drive_root.exists() else []
    raise FileNotFoundError(
        "Could not find the repository in Google Drive.\n"
        f"Looked for '{REPO_NAME}/artifacts/ARTIFACT_PROVENANCE.csv' (up to 4 folders deep) and for '{REPO_NAME}*.zip' in {drive_root}.\n"
        f"Items in My Drive whose names look related: {near or 'none'}.\n"
        "Fix: upload the zip file (or the unzipped folder) to My Drive, or set DRIVE_REPO to the folder that "
        "contains README.md, artifacts/ and notebooks/.")


_previous = globals().get("REPO_ROOT")                 # set by an earlier notebook in the same runtime (R00)
if _previous is not None and _is_repo(Path(_previous)):
    REPO_ROOT = Path(_previous)
elif IN_COLAB and REPO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = _locate_in_drive(Path(DRIVE_REPO), Path("/content/drive/MyDrive"))
elif IN_COLAB:
    target = Path("/content/" + REPO_NAME)
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(target)], check=True)
    REPO_ROOT = _find_repo(target)
else:                                     # local Jupyter: the notebook lives in notebooks/reproduction/
    REPO_ROOT = _find_repo(Path.cwd().resolve())
os.chdir(REPO_ROOT)

# The results were verified with these exact versions (the versions recorded by the Colab analysis).
PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.16.3", "scikit-learn": "1.6.1",
          "matplotlib": "3.10.0", "pytest": "8.3.5"}
wrong = {k: v for k, v in PINNED.items() if version(k) != v}
if wrong and IN_COLAB:
    print("Installing the pinned versions:", wrong)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
    print("Done. The runtime will now restart; then run the notebook again from the top (Runtime > Run all).")
    os.kill(os.getpid(), 9)
elif wrong:
    print("WARNING: library versions differ from requirements.txt:", wrong)
print("Repository:", REPO_ROOT)
print("Python", sys.version.split()[0], "|", ", ".join(f"{k} {version(k)}" for k in PINNED))

## Parameters

In [ ]:
# ===== Parameters =====
# Leave ARCHIVES = None to use only the files in the repository (about one minute).
# To re-derive every archive-based table from the 36 prediction archives, set ARCHIVES to the Drive folder
# that holds them, e.g. "/content/drive/MyDrive/prediction_archives" (takes 20-40 minutes in Colab).
ARCHIVES = None
WORKERS = 1      # parallel processes for the archives; keep 1 on a standard Colab runtime (memory)

## Library code
Verbatim copies of the repository modules this notebook uses (`src/`). Run these cells; there is nothing to edit.

In [ ]:
# ===== src/utils/hashing.py =====
"""Hashing utilities that reproduce the identity conventions of the frozen Experiment 1 v2 artefacts.

Two conventions were recovered from the artefacts themselves and are checked by
``tests/test_manifest_hashes.py``:

* ``config_hash`` = SHA-256 of the canonical JSON of ``effective_config``.
* Manifest self-hashes (``partition_policy_hash``, ``selected_models_manifest_hash``,
  ``manifest_hash``, ``diagnostic_manifest_hash``, ``implementation_note_hash``, ``policy_hash``)
  = SHA-256 of the canonical JSON of the manifest without the hash field itself and without
  ``created_at_utc``.

Canonical JSON means ``json.dumps(obj, sort_keys=True, separators=(",", ":"))`` encoded as UTF-8.

Command line::

    python -m src.utils.hashing sha256 FILE [FILE ...]
    python -m src.utils.hashing manifest DIRECTORY > SHA256SUMS.txt
"""

import argparse
import hashlib
import json
from pathlib import Path
from typing import Iterable

EXCLUDED_FROM_SELF_HASH = ("created_at_utc",)


def sha256_file(path: str | Path, chunk: int = 1 << 20) -> str:
    """Return the hexadecimal SHA-256 digest of a file's bytes."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def canonical_json(obj) -> str:
    """Canonical JSON text used by the Experiment 1 v2 notebooks."""
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))


def canonical_json_hash(obj) -> str:
    """SHA-256 of the canonical JSON text of ``obj``."""
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()


def manifest_self_hash(manifest: dict, hash_field: str,
                       excluded: Iterable[str] = EXCLUDED_FROM_SELF_HASH) -> str:
    """Recompute a manifest's self-hash: canonical JSON without ``hash_field`` and ``excluded`` keys."""
    drop = set(excluded) | {hash_field}
    return canonical_json_hash({k: v for k, v in manifest.items() if k not in drop})


def config_hash(effective_config: dict) -> str:
    """Recompute ``config_hash`` from an ``effective_config`` block."""
    return canonical_json_hash(effective_config)

In [ ]:
# ===== src/utils/io_utils.py =====
"""Paths and loaders shared by the reproduction scripts and tests."""

import json
import os
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(REPO_ROOT)  # set by the setup cell
ARTIFACTS = REPO_ROOT / "artifacts"
RESULTS = REPO_ROOT / "results"
DATASETS = {"cicids2017": "CICIDS2017", "unsw_nb15": "UNSW-NB15"}

# Columns of every observation-level prediction archive written by NB-03/NB-07.
ARCHIVE_COLUMNS = ["row_id", "y_true", "reconstruction_error", "threshold", "y_pred", "model", "seed",
                   "config_hash", "run_hash", "attack_family", "feature_fingerprint"]
ARCHIVE_DTYPES = {"row_id": str, "y_true": "int8", "reconstruction_error": "float64", "threshold": "float64",
                  "y_pred": "int8", "model": str, "seed": "int64", "config_hash": str, "run_hash": str,
                  "attack_family": str, "feature_fingerprint": str}
# CICIDS2017 web-attack family names contain U+FFFD where the original CSVs had a cp1252 en dash
# ("Web Attack � Brute Force"); the character is kept as stored in the archives.

# Environment variable that points to the folder holding the external prediction archives.
ARCHIVE_ENV = "NIDS_PREDICTION_ARCHIVES"


def art(*parts: str) -> Path:
    """Path inside ``artifacts/``."""
    return ARTIFACTS.joinpath(*parts)


def load_json(path: str | Path) -> dict:
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def read_csv(path: str | Path, **kw) -> pd.DataFrame:
    """``pandas.read_csv`` with exact (round-trip) float parsing, so that committed tables compare exactly."""
    kw.setdefault("float_precision", "round_trip")
    return pd.read_csv(path, **kw)


def read_prediction_archive(path: str | Path) -> pd.DataFrame:
    """Read one observation-level prediction archive with the dtypes used in the study."""
    return pd.read_csv(path, usecols=ARCHIVE_COLUMNS, dtype=ARCHIVE_DTYPES)


def archive_dir(cli_value: str | None = None) -> Path | None:
    """Folder with the external prediction archives (CLI argument first, then the environment variable)."""
    value = cli_value or os.environ.get(ARCHIVE_ENV)
    if not value:
        return None
    p = Path(value).expanduser()
    return p if p.is_dir() else None


def dataset_of_row_id(row_id: str) -> str:
    """``row_id`` is ``<dataset>::<source file>::<10-digit source row>``; return the repository dataset key."""
    prefix = row_id.split("::", 1)[0]
    return {"CICIDS2017": "cicids2017", "UNSWNB15": "unsw_nb15"}[prefix]


def role_sizes(ds: str) -> dict:
    """Evaluation-role sizes recorded in the frozen result tables (TP+FP+FN+TN of any run)."""
    fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv")).iloc[0]
    dev = read_csv(art(ds, "model_selection", "confirmation_seed_results.csv")).iloc[0]
    tot = lambda r: int(r.TP + r.FP + r.FN + r.TN)
    return {"final_test": tot(fin), "development": tot(dev)}


def write_csv(df: pd.DataFrame, path: str | Path, **kw) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator="\n", **kw)
    return path

In [ ]:
# ===== src/utils/seeds.py =====
"""Seed register of Experiment 1 v2.

Every constant below is read back from a frozen artefact by ``tests/test_seed_register.py``;
the module does not set any random state for model training (training ran in the Colab notebooks).
"""
FINAL_SEEDS = (101, 202, 303, 404, 505)          # final_seed_results.csv
CONFIRMATION_SEEDS = (101, 202, 303)             # selected_models_manifest.json["confirmation_seeds"]
PARTITION_SEED = 20260914                        # partition_policy_manifest.json / integrity reports
BOOTSTRAP_SEED = 2026                            # paired_seed_bootstrap_effects.csv["bootstrap_seed"]
BOOTSTRAP_ITERATIONS = 10_000                    # paired_seed_bootstrap_effects.csv["bootstrap_iterations"]
CONFIDENCE_LEVEL = 0.95                          # paired_seed_bootstrap_effects.csv["confidence_level"]

In [ ]:
# ===== src/utils/claims.py =====
"""Record manuscript claims next to the values recomputed from the artefacts.

Each check compares a recomputed number with the value *as printed in the manuscript*. The tolerance is
half a unit in the last printed decimal place, so "0.7094" passes for any value in [0.70935, 0.70945].
Every reproduction script writes its log to ``results/claims_check/<script>.csv`` and exits with a
non-zero status if any check fails.
"""

import math
import sys
from pathlib import Path

import pandas as pd



def _parse(printed: str) -> tuple[float, float]:
    s = str(printed).replace(",", "").replace("%", "").replace("+", "").replace("−", "-").strip()
    dec = len(s.split(".")[1]) if "." in s and "e" not in s.lower() else 0
    if "e" in s.lower():
        mant = s.lower().split("e")[0]
        exp = int(s.lower().split("e")[1])
        dec_m = len(mant.split(".")[1]) if "." in mant else 0
        return float(s), 0.5 * 10 ** (exp - dec_m)
    return float(s), 0.5 * 10 ** (-dec)


class ClaimLog:
    def __init__(self, script: str):
        self.script = script
        self.rows: list[dict] = []

    def check(self, claim_id: str, location: str, description: str, reproduced, printed: str,
              source: str, scale: float = 1.0) -> bool:
        """``scale`` converts the reproduced value to the printed unit (e.g. 100 for percentages)."""
        value, tol = _parse(printed)
        rep = float(reproduced) * scale
        ok = math.isclose(rep, value, abs_tol=tol + 1e-12)
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": printed, "reproduced_value": repr(round(rep, 10)),
                          "status": "PASS" if ok else "FAIL", "source": source})
        return ok

    def assert_true(self, claim_id: str, location: str, description: str, condition: bool, source: str) -> bool:
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "true", "reproduced_value": str(bool(condition)),
                          "status": "PASS" if condition else "FAIL", "source": source})
        return bool(condition)

    def note(self, claim_id: str, location: str, description: str, value: str, source: str, status: str) -> None:
        """Record a finding that is not a pass/fail comparison (e.g. a discrepancy that the manuscript notes)."""
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "", "reproduced_value": value, "status": status, "source": source})

    def finish(self) -> int:
        df = pd.DataFrame(self.rows)
        out = RESULTS / "claims_check" / f"{self.script}.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out, index=False, lineterminator="\n")
        n_fail = int((df.status == "FAIL").sum())
        n_pass = int((df.status == "PASS").sum())
        print(f"[{self.script}] {n_pass} checks passed, {n_fail} failed, {len(df) - n_pass - n_fail} notes -> {out}")
        if n_fail:
            print(df[df.status == "FAIL"].to_string(index=False), file=sys.stderr)
        return 1 if n_fail else 0

In [ ]:
# ===== src/evaluation/metrics.py =====
"""Observation-level detection metrics, named as in the frozen ``final_seed_results.csv`` tables.

Positive class = attack (``y_true == 1``). Scores are reconstruction errors; a larger score is more
anomalous. ``PR-AUC`` is the trapezoidal area under the precision-recall curve
(``sklearn.metrics.auc(recall, precision)``), which is distinct from ``Average_Precision``.
Each function was checked against every frozen run in the study (see ``tests/test_archive_recomputation.py``).
"""

import numpy as np
from sklearn.metrics import (auc, average_precision_score, precision_recall_curve, roc_auc_score,
                             roc_curve)


def confusion(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y = np.asarray(y_true).astype(bool)
    p = np.asarray(y_pred).astype(bool)
    return {"TP": int((y & p).sum()), "FP": int((~y & p).sum()),
            "FN": int((y & ~p).sum()), "TN": int((~y & ~p).sum())}


def mcc_from_counts(TP: int, FP: int, FN: int, TN: int) -> float:
    den = np.sqrt(float(TP + FP) * float(TP + FN) * float(TN + FP) * float(TN + FN))
    return float((TP * TN - FP * FN) / den) if den > 0 else 0.0


def threshold_metrics(TP: int, FP: int, FN: int, TN: int) -> dict:
    """Threshold-dependent metrics from a confusion matrix."""
    rec = TP / (TP + FN)
    prec = TP / (TP + FP)
    fpr = FP / (FP + TN)
    return {"MCC": mcc_from_counts(TP, FP, FN, TN), "Recall": rec,
            "F1": 2 * prec * rec / (prec + rec), "Precision": prec, "FPR": fpr,
            "FP_per_1000_benign": 1000.0 * fpr, "Balanced_Accuracy": (rec + TN / (TN + FP)) / 2}


def ranking_metrics(y_true: np.ndarray, scores: np.ndarray) -> dict:
    """Threshold-free metrics of the reconstruction-error ranking."""
    pr, rc, _ = precision_recall_curve(y_true, scores)
    fpr, tpr, _ = roc_curve(y_true, scores)
    return {"PR-AUC": float(auc(rc, pr)), "ROC-AUC": float(roc_auc_score(y_true, scores)),
            "Average_Precision": float(average_precision_score(y_true, scores)),
            "TPR_at_1pct_FPR": float(tpr[fpr <= 0.01].max())}


def all_metrics(y_true: np.ndarray, y_pred: np.ndarray, scores: np.ndarray) -> dict:
    c = confusion(y_true, y_pred)
    out = dict(c)
    out.update(threshold_metrics(**c))
    out.update(ranking_metrics(y_true, scores))
    return out


# Metric columns shared by final_seed_results.csv and confirmation_seed_results.csv
FROZEN_METRIC_COLUMNS = ["MCC", "Recall", "F1", "Precision", "PR-AUC", "FPR", "FP_per_1000_benign",
                         "Balanced_Accuracy", "ROC-AUC", "Average_Precision", "TPR_at_1pct_FPR",
                         "TP", "TN", "FP", "FN"]

In [ ]:
# ===== src/evaluation/thresholding.py =====
"""Checks of the frozen decision rule in the prediction archives.

The decision rule recorded in every archive is ``y_pred = 1`` if and only if
``reconstruction_error > threshold`` (strict inequality), with one threshold per run. The threshold
itself is the 99th percentile of benign calibration-role reconstruction error; the calibration
errors are not part of the released evidence, so only the rule and its constancy are checked here.
The ladder check in ``tests/test_partition_accounting.py`` confirms, for the UNSW-NB15 v3.4 gate,
that the recorded threshold equals the recorded calibration 0.99 quantile.
"""

import numpy as np
import pandas as pd


def apply_threshold(scores: np.ndarray, threshold: float) -> np.ndarray:
    """Frozen rule: flag an observation as attack when its error is strictly above the threshold."""
    return (np.asarray(scores) > threshold).astype(np.int8)


def check_archive_rule(df: pd.DataFrame) -> dict:
    """Return the integrity flags of one prediction archive."""
    thr = df["threshold"].to_numpy()
    unique = np.unique(thr)
    consistent = bool((apply_threshold(df["reconstruction_error"].to_numpy(), unique[0]) ==
                       df["y_pred"].to_numpy()).all()) if unique.size == 1 else False
    return {"threshold_unique_values": int(unique.size), "threshold": float(unique[0]),
            "prediction_rule_consistent": consistent, "row_id_unique": bool(df["row_id"].is_unique)}

In [ ]:
# ===== src/evaluation/paired_analysis.py =====
"""Paired seed-level contrasts (DAE minus AE) used for Table 10 and Fig. 8 of the manuscript.

The resampling unit is the seed pair. With five pairs there are 5**5 = 3,125 equally likely ordered
bootstrap resamples, so the bootstrap distribution of the mean can be enumerated exactly; the
manuscript reports percentile limits of that exact distribution (NumPy default linear interpolation).
``bootstrap_interval`` is a Monte-Carlo percentile bootstrap with ``numpy.random.default_rng(2026)`` and
10,000 resamples, kept for comparison. It is *not* the NB-08 code (which is not in the evidence set): at six
decimals it equals the exact limits for 31 of 32 bounds (CICIDS2017 PR-AUC upper: -0.007701 vs -0.007694),
whereas the frozen NB-08 table equals them for 30 of 32 (see docs/LIMITATIONS.md).
"""

import itertools

import numpy as np
import pandas as pd
from scipy import stats



def paired_differences(values: pd.DataFrame, metric: str) -> pd.Series:
    """``values`` has columns metric, seed, AE, DAE (frozen ``paired_seed_values.csv`` layout)."""
    v = values[values.metric == metric].sort_values("seed")
    return pd.Series((v.DAE - v.AE).to_numpy(), index=v.seed.to_numpy(), name=metric)


def exact_bootstrap_means(d: np.ndarray) -> np.ndarray:
    """Means of all n**n ordered resamples with replacement of the paired differences."""
    d = np.asarray(d, dtype=float)
    idx = np.array(list(itertools.product(range(d.size), repeat=d.size)))
    return d[idx].mean(axis=1)


def exact_interval(d: np.ndarray, level: float = CONFIDENCE_LEVEL) -> tuple[float, float]:
    a = (1 - level) / 2
    lo, hi = np.percentile(exact_bootstrap_means(d), [100 * a, 100 * (1 - a)])
    return float(lo), float(hi)


def bootstrap_interval(d: np.ndarray, iterations: int = BOOTSTRAP_ITERATIONS, seed: int = BOOTSTRAP_SEED,
                       level: float = CONFIDENCE_LEVEL) -> tuple[float, float]:
    """Monte-Carlo percentile bootstrap over seed pairs with ``numpy.random.default_rng(seed)``."""
    d = np.asarray(d, dtype=float)
    rng = np.random.default_rng(seed)
    means = d[rng.integers(0, d.size, size=(iterations, d.size))].mean(axis=1)
    a = (1 - level) / 2
    lo, hi = np.percentile(means, [100 * a, 100 * (1 - a)])
    return float(lo), float(hi)


def summarise(d: np.ndarray) -> dict:
    """Descriptive and inferential summary of one vector of paired differences."""
    d = np.asarray(d, dtype=float)
    n = d.size
    mean, sd = d.mean(), d.std(ddof=1)
    sign = np.sign(mean)
    t = stats.ttest_1samp(d, 0.0)
    t_lo, t_hi = stats.t.interval(CONFIDENCE_LEVEL, n - 1, loc=mean, scale=sd / np.sqrt(n))
    lo, hi = exact_interval(d)
    return {"n_pairs": n, "mean_diff": mean, "median_diff": float(np.median(d)), "sd_diff": sd,
            "dz": mean / sd, "n_same_sign": int((np.sign(d) == sign).sum()),
            "t": float(t.statistic), "p_t": float(t.pvalue), "t_lo": float(t_lo), "t_hi": float(t_hi),
            "p_sign": float(stats.binomtest(int((d > 0).sum()), int((d != 0).sum())).pvalue),
            "p_wilcoxon": float(stats.wilcoxon(d).pvalue),
            "exact_lo": lo, "exact_hi": hi}


def contrast_table(values: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    rows = []
    for m in metrics:
        d = paired_differences(values, m).to_numpy()
        rows.append({"metric": m, **summarise(d)})
    return pd.DataFrame(rows)

In [ ]:
# ===== src/evaluation/family_diagnostics.py =====
"""Attack-family diagnostics of the final-test runs (Table S3 of the Supplementary Material).

Family figures are descriptive. They played no part in model selection
(``selected_models_manifest.json["family_metrics_used_for_selection"] is False``).
"""

import pandas as pd


def family_table(fam: pd.DataFrame) -> pd.DataFrame:
    """Pool the five final seeds per model and family.

    ``fam`` is ``family_detection_final_test.csv``. Recall per row = detected rows / rows; recall per
    vector = detected fingerprints / fingerprints (a fingerprint counts as detected when its rows are
    flagged; every fingerprint group received one prediction); miss share = the family's share of the
    model's missed attack rows.
    """
    g = fam.groupby(["model", "attack_family"]).agg(rows=("rows", "sum"), det=("detected_rows", "sum"),
                                                    fps=("fingerprints", "sum"), detfp=("detected_fingerprints", "sum"),
                                                    seeds=("seed", "nunique"), rows_per_seed=("rows", "max"),
                                                    fps_per_seed=("fingerprints", "max")).reset_index()
    g["recall_per_row"] = g.det / g.rows
    g["recall_per_vector"] = g.detfp / g.fps
    g["misses"] = g.rows - g.det
    g["miss_share_pct"] = 100 * g.misses / g.groupby("model").misses.transform("sum")
    wide = g.pivot(index="attack_family", columns="model",
                   values=["recall_per_row", "recall_per_vector", "miss_share_pct"])
    wide.columns = [f"{m}_{v}" for v, m in wide.columns]
    base = g[g.model == "AE"].set_index("attack_family")[["rows_per_seed", "fps_per_seed", "seeds"]]
    out = base.join(wide).reset_index().rename(columns={"rows_per_seed": "rows", "fps_per_seed": "distinct_vectors"})
    return out.sort_values("rows", ascending=False).reset_index(drop=True)

In [ ]:
# ===== src/evaluation/archive_summary.py =====
"""Summarise one observation-level prediction archive into the per-run quantities used in the manuscript.

Produces, for one archive:
* a run record: integrity flags, confusion counts, all frozen metrics, fingerprint-level recall and FPR,
  score geometry of missed attacks, and the largest fingerprint group present;
* per-family detection counts, per row and per distinct attack fingerprint;
* label counts of every mixed-label fingerprint group present (development archives only).

Used by ``scripts/reproduce_final_statistics.py --archives DIR``. The outputs committed under
``artifacts/<dataset>/*/derived_from_archives/`` were produced by this module from the 36 archives.
"""

from pathlib import Path

import numpy as np
import pandas as pd



def summarise_archive(path: str | Path, with_sha256: bool = True) -> dict:
    path = Path(path)
    df = read_prediction_archive(path)
    ds = dataset_of_row_id(df.row_id.iat[0])
    sizes = role_sizes(ds)
    n = len(df)
    role = {v: k for k, v in sizes.items()}.get(n, "unknown")
    y = df.y_true.to_numpy()
    p = df.y_pred.to_numpy()
    s = df.reconstruction_error.to_numpy()

    run = {"archive_file": path.name, "dataset": ds, "role": role, "model": df.model.iat[0],
           "seed": int(df.seed.iat[0]), "config_hash": df.config_hash.iat[0], "run_hash": df.run_hash.iat[0],
           "n_rows": n, "n_attack_rows": int(y.sum())}
    if with_sha256:
        run["archive_sha256"] = sha256_file(path)
        run["archive_bytes"] = path.stat().st_size
    run.update(check_archive_rule(df))
    run.update(all_metrics(y, p, s))

    # fingerprint level
    g = df.groupby("feature_fingerprint", sort=False).agg(
        lab_min=("y_true", "min"), lab_max=("y_true", "max"),
        pmin=("y_pred", "min"), pmax=("y_pred", "max"), size=("y_true", "size"))
    att = g[g.lab_min == 1]
    ben = g[g.lab_max == 0]
    run.update({"n_fingerprints": len(g), "n_mixed_label_fingerprints": int((g.lab_min != g.lab_max).sum()),
                "n_fingerprints_with_varying_prediction": int((g.pmin != g.pmax).sum()),
                "n_attack_fingerprints": len(att), "recall_per_attack_fingerprint": float((att.pmax == 1).mean()),
                "n_benign_fingerprints": len(ben), "fpr_per_benign_fingerprint": float((ben.pmax == 1).mean()),
                "largest_fingerprint_group_rows": int(g["size"].max())})
    nfam = df[df.y_true == 1].groupby("feature_fingerprint").attack_family.nunique()
    multi = nfam.index[nfam > 1]
    run["n_attack_fingerprints_with_several_families"] = int(len(multi))
    run["attack_rows_in_several_family_fingerprints"] = int(df.feature_fingerprint.isin(multi).sum())
    big = g["size"].idxmax()
    fam_big = df.loc[df.feature_fingerprint == big, "attack_family"].dropna().unique()
    run["largest_fingerprint_group_families"] = "|".join(sorted(fam_big)) if len(fam_big) else "BENIGN"

    # score geometry relative to the run threshold
    thr = run["threshold"]
    miss = s[(y == 1) & (p == 0)]
    ben_median = float(np.median(s[y == 0]))
    run.update({"benign_median_error": ben_median,
                "missed_attack_median_error_over_threshold": float(np.median(miss) / thr) if miss.size else np.nan,
                "missed_attack_p90_error_over_threshold": float(np.percentile(miss, 90) / thr) if miss.size else np.nan,
                "attack_share_below_benign_median": float((s[y == 1] < ben_median).mean()),
                "attack_share_at_or_below_threshold": float((s[y == 1] <= thr).mean())})

    # families (attack rows only)
    a = df[df.y_true == 1]
    fam_rows = a.groupby("attack_family").agg(rows=("y_pred", "size"), detected_rows=("y_pred", "sum"))
    fam_fp = a.groupby(["attack_family", "feature_fingerprint"]).y_pred.max().groupby("attack_family").agg(
        fingerprints="size", detected_fingerprints="sum")
    fam = fam_rows.join(fam_fp).reset_index()
    for k in ("dataset", "role", "model", "seed", "config_hash", "run_hash"):
        fam.insert(0, k, run[k])
    fam = fam[["dataset", "role", "model", "seed", "config_hash", "run_hash", "attack_family",
               "rows", "detected_rows", "fingerprints", "detected_fingerprints"]]

    # mixed-label groups with their label composition
    mixed_ids = g.index[g.lab_min != g.lab_max]
    m = df[df.feature_fingerprint.isin(mixed_ids)].assign(
        detected_attack=lambda x: ((x.y_true == 1) & (x.y_pred == 1)).astype(int))
    mixed = m.groupby("feature_fingerprint").agg(rows=("y_true", "size"), attack_rows=("y_true", "sum"),
                                                 attack_rows_detected=("detected_attack", "sum"))
    mixed["benign_rows"] = mixed.rows - mixed.attack_rows
    fams = m[m.y_true == 1].groupby("feature_fingerprint").attack_family.agg(lambda v: "|".join(sorted(set(v.dropna()))))
    mixed["attack_families"] = fams
    mixed = mixed.reset_index()[["feature_fingerprint", "rows", "benign_rows", "attack_rows", "attack_families",
                                 "attack_rows_detected"]]
    keep = ~df.feature_fingerprint.isin(mixed_ids).to_numpy()
    run["MCC_excluding_mixed_label_groups"] = mcc_from_counts(**confusion(y[keep], p[keep]))
    run["mixed_label_rows"] = int((~keep).sum())
    run["mixed_label_attack_rows"] = int(y[~keep].sum())
    run["mixed_label_attack_rows_detected"] = int(p[(~keep) & (y == 1)].sum())
    return {"run": run, "families": fam, "mixed_groups": mixed}


def summarise_directory(archive_dir: str | Path, out_dir: str | Path, workers: int = 1) -> dict:
    """Summarise every ``*.csv`` archive under ``archive_dir`` and write tidy tables to ``out_dir``.

    Writes ``run_summary.csv`` (one row per archive), ``family_detection.csv`` and
    ``mixed_label_groups.csv`` (one row per mixed-label group per archive). Returns the three tables.
    """
    from concurrent.futures import ProcessPoolExecutor
    files = sorted(Path(archive_dir).glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"no prediction archives (*.csv) found in {archive_dir}")
    if workers > 1:
        with ProcessPoolExecutor(workers) as ex:
            parts = list(ex.map(summarise_archive, files))
    else:
        parts = [summarise_archive(f) for f in files]
    runs = pd.DataFrame([p["run"] for p in parts]).sort_values(["dataset", "role", "model", "config_hash", "seed"])
    fams = pd.concat([p["families"] for p in parts]).sort_values(["dataset", "role", "model", "config_hash", "seed", "attack_family"])
    mixed = []
    for p in parts:
        if len(p["mixed_groups"]):
            r = p["run"]
            mixed.append(p["mixed_groups"].assign(dataset=r["dataset"], role=r["role"], model=r["model"],
                                                  seed=r["seed"], config_hash=r["config_hash"]))
    mixed = pd.concat(mixed) if mixed else pd.DataFrame()
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    runs.to_csv(out / "run_summary.csv", index=False, lineterminator="\n")
    fams.to_csv(out / "family_detection.csv", index=False, lineterminator="\n")
    mixed.to_csv(out / "mixed_label_groups.csv", index=False, lineterminator="\n")
    return {"runs": runs, "families": fams, "mixed": mixed}


def repository_tables(tables: dict) -> dict:
    """Split the directory summary into the per-dataset files committed under ``artifacts/``.

    Keys are paths relative to ``artifacts/``. Mixed-label group composition is identical in every
    development archive of a dataset (checked here), so it is stored once per dataset.
    """
    runs, fams, mixed = tables["runs"], tables["families"], tables["mixed"]
    out = {}
    for ds in sorted(runs.dataset.unique()):
        r = runs[runs.dataset == ds]
        f = fams[fams.dataset == ds]
        out[f"{ds}/final_evaluation/derived_from_archives/run_summary_final_test.csv"] = r[r.role == "final_test"]
        out[f"{ds}/final_evaluation/derived_from_archives/family_detection_final_test.csv"] = f[f.role == "final_test"]
        out[f"{ds}/model_selection/derived_from_archives/run_summary_development.csv"] = r[r.role == "development"]
        if len(mixed):
            mx = mixed[mixed.dataset == ds]
            comp = mx[["feature_fingerprint", "rows", "benign_rows", "attack_rows", "attack_families", "role"]]
            base = comp.drop_duplicates()
            if base.feature_fingerprint.duplicated().any():
                raise ValueError(f"{ds}: mixed-label group composition differs between archives")
            out[f"{ds}/fingerprints/derived_from_archives/mixed_label_group_composition.csv"] = \
                base.sort_values("feature_fingerprint").reset_index(drop=True)
    manifest = runs[["archive_file", "archive_sha256", "archive_bytes", "n_rows", "dataset", "role", "model",
                     "seed", "config_hash", "run_hash"]]
    out["../external_artifacts/prediction_archive_manifest.csv"] = manifest.sort_values(
        ["dataset", "role", "model", "config_hash", "seed"])
    return out


def write_repository_tables(tables: dict, root: str | Path) -> list[Path]:
    """Write ``repository_tables(tables)`` below ``root`` (normally the ``artifacts/`` folder)."""
    written = []
    for rel, df in repository_tables(tables).items():
        p = (Path(root) / rel).resolve()
        p.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(p, index=False, lineterminator="\n", float_format="%.17g")
        written.append(p)
    return written

## Reproduction: `scripts/reproduce_final_statistics.py`

In [ ]:
import argparse
import sys
from pathlib import Path


import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402


class A:                     # values from the parameter cell (replaces the command-line arguments)
    archives = ARCHIVES
    workers = WORKERS

L = ClaimLog("reproduce_final_statistics")
DS = {"cicids2017": "CICIDS2017", "unsw_nb15": "UNSW-NB15"}
METRICS = ["MCC", "Recall", "F1", "Precision", "Balanced_Accuracy", "PR-AUC", "FPR", "FP_per_1000_benign"]
HIER = [("mean_MCC", False), ("mean_FPR", True), ("mean_PR_AUC", False), ("MCC_SD", True),
        ("parameter_count", True), ("mean_training_duration_seconds", True)]

### 0. optional: re-derive the archive summaries and compare with the committed tables

In [ ]:
adir = archive_dir(A.archives)
if A.archives and adir is None:
    sys.exit(f"--archives {A.archives}: folder not found")
if adir is not None:
    print(f"re-deriving archive summaries from {adir} with {A.workers} worker(s) ...")
    tabs = summarise_directory(adir, RESULTS / "archive_recomputation", workers=A.workers)
    for rel, new in repository_tables(tabs).items():
        old = read_csv((ARTIFACTS / rel).resolve())
        new = new.reset_index(drop=True)
        same_shape = list(old.columns) == list(new.columns) and len(old) == len(new)
        num_ok = same_shape and all(
            np.allclose(old[c].to_numpy(float), new[c].to_numpy(float), rtol=0, atol=1e-12, equal_nan=True)
            if pd.api.types.is_numeric_dtype(old[c]) else (old[c].astype(str).to_numpy() == new[c].astype(str).to_numpy()).all()
            for c in old.columns)
        L.assert_true(f"archives.{rel}", "Supplementary S1 (verification)", f"re-derived {rel} equals the committed table", num_ok,
                      "prediction archives")
    man = read_csv(REPO_ROOT / "external_artifacts" / "prediction_archive_manifest.csv")
    got = tabs["runs"][["archive_file", "archive_sha256"]].merge(man[["archive_file", "archive_sha256"]], on="archive_file", how="outer",
                                                                 suffixes=("", "_manifest"))
    L.assert_true("archives.sha256", "Data availability", "every archive's SHA-256 equals the deposited manifest (36 files)",
                  len(got) == 36 and bool((got.archive_sha256 == got.archive_sha256_manifest).all()), "prediction_archive_manifest.csv")

### 1. model selection (Table 9; Section 4.7 first paragraph)

In [ ]:
cfg_rows, rank_rows = [], []
for ds, name in DS.items():
    sel = load_json(art(ds, "model_selection", "selected_models_manifest.json"))
    conf = read_csv(art(ds, "model_selection", "confirmation_seed_results.csv"))
    ranking = read_csv(art(ds, "model_selection", "selection_ranking.csv"))
    L.assert_true(f"sel.{ds}.family_not_used", "Sec. 4.7", "family metrics not used for selection", sel["family_metrics_used_for_selection"] is False,
                  "selected_models_manifest.json")
    L.assert_true(f"sel.{ds}.seeds", "Sec. 4.7", "confirmation seeds 101, 202, 303", tuple(sel["confirmation_seeds"]) == CONFIRMATION_SEEDS,
                  "selected_models_manifest.json")
    # recompute the confirmation summary and the selection hierarchy
    s = conf.groupby(["model_type", "config_hash"]).agg(
        mean_MCC=("MCC", "mean"), median_MCC=("MCC", "median"), MCC_SD=("MCC", "std"),
        MCC_range=("MCC", lambda v: v.max() - v.min()), mean_FPR=("FPR", "mean"),
        mean_FP_per_1000_benign=("FP_per_1000_benign", "mean"), mean_PR_AUC=("PR-AUC", "mean"),
        parameter_count=("parameter_count", "first"), mean_training_duration_seconds=("training_duration_seconds", "mean"),
        n_seeds=("seed", "nunique")).reset_index()
    s = s.sort_values(["model_type"] + [h for h, _ in HIER], ascending=[True] + [a for _, a in HIER])
    s["selection_rank"] = s.groupby("model_type").cumcount() + 1
    merged = s.merge(ranking, on=["model_type", "config_hash"], suffixes=("", "_frozen"))
    cols = ["mean_MCC", "median_MCC", "MCC_SD", "MCC_range", "mean_FPR", "mean_FP_per_1000_benign", "mean_PR_AUC",
            "mean_training_duration_seconds", "selection_rank"]
    L.assert_true(f"sel.{ds}.ranking", "Methods (selection hierarchy)", "selection ranking recomputed from confirmation seeds equals the frozen ranking",
                  len(merged) == len(ranking) == len(s) and all(np.allclose(merged[c], merged[c + "_frozen"], rtol=0, atol=1e-12) for c in cols),
                  "confirmation_seed_results.csv vs selection_ranking.csv")
    for m in ("AE", "DAE"):
        blk = sel[f"selected_{m}"]
        top = s[(s.model_type == m) & (s.selection_rank == 1)].iloc[0]
        L.assert_true(f"sel.{ds}.{m}.rank1", "Table 9", f"rank-1 {m} configuration is the selected one", top.config_hash == blk["config_hash"],
                      "selected_models_manifest.json")
        L.assert_true(f"sel.{ds}.{m}.config_hash", "Table 9", f"{m} config_hash = SHA-256 of canonical effective_config",
                      config_hash(blk["effective_config"]) == blk["config_hash"], "selected_models_manifest.json")
        e = blk["effective_config"]
        cfg_rows.append({"dataset": name, "model": m, "architecture": e["architecture"], "latent_dim": e["latent_dim"],
                         "training_loss": e["training_loss"], "learning_rate": e["learning_rate"], "batch_size": e["batch_size"],
                         "weight_decay": e["weight_decay"], "numerical_masking": e["numerical_masking"] if m == "DAE" else None,
                         "grouped_categorical_masking": e.get("grouped_categorical_masking"),
                         "hidden_activation": e["hidden_activation"], "optimizer": e["optimizer"], "gradient_clip_norm": e["gradient_clip_norm"],
                         "maximum_epochs": e["maximum_epochs"], "early_stopping_patience": e["early_stopping_patience"],
                         "parameter_count": int(top.parameter_count), "dev_mean_MCC": top.mean_MCC, "dev_MCC_SD": top.MCC_SD,
                         "mean_training_seconds": top.mean_training_duration_seconds, "config_hash": blk["config_hash"]})
    rank_rows.append(s.assign(dataset=name))
cfg = pd.DataFrame(cfg_rows)
T9 = {("CICIDS2017", "AE"): ("4.79e-4", "128", "15,390", "0.530234", "0.004590", "2,113", "b86121d8"),
      ("CICIDS2017", "DAE"): ("3.66e-4", "256", "15,390", "0.524699", "0.001429", "1,288", "16838232"),
      ("UNSW-NB15", "AE"): ("1.07e-4", "512", "15,519", "0.903582", "0.003120", "970", "f2ff4491"),
      ("UNSW-NB15", "DAE"): ("5.52e-4", "128", "15,519", "0.906182", "0.000240", "4,708", "efc7b06f")}
for (dsn, m), (lr, bs, pc, mcc, sd, tr, h) in T9.items():
    r = cfg[(cfg.dataset == dsn) & (cfg.model == m)].iloc[0]
    k = f"T9.{dsn}.{m}"
    L.check(k + ".lr", "Table 9", "learning rate", r.learning_rate, lr, "selected_models_manifest.json")
    L.check(k + ".batch", "Table 9", "batch size", r.batch_size, bs, "selected_models_manifest.json")
    L.check(k + ".params", "Table 9", "parameter count", r.parameter_count, pc, "confirmation_seed_results.csv")
    L.check(k + ".dev_mcc", "Table 9", "development mean MCC (3 seeds)", r.dev_mean_MCC, mcc, "confirmation_seed_results.csv")
    L.check(k + ".dev_sd", "Table 9", "development MCC SD (3 seeds)", r.dev_MCC_SD, sd, "confirmation_seed_results.csv")
    L.check(k + ".train_s", "Table 9", "mean training seconds", r.mean_training_seconds, tr, "confirmation_seed_results.csv")
    L.assert_true(k + ".hash", "Table 9", "config hash prefix", r.config_hash.startswith(h), "selected_models_manifest.json")
L.assert_true("T9.common", "Table 9 caption", "all selected models: architecture B, latent 16, ReLU, AdamW, clip 1.0, <=150 epochs, patience 12",
              bool(((cfg.architecture == "B") & (cfg.latent_dim == 16) & (cfg.hidden_activation == "ReLU") & (cfg.optimizer == "AdamW")
                    & (cfg.gradient_clip_norm == 1.0) & (cfg.maximum_epochs == 150) & (cfg.early_stopping_patience == 12)).all()),
              "selected_models_manifest.json")
L.assert_true("T9.losses_masks", "Table 9", "losses and masking rates as tabulated",
              list(cfg.training_loss) == ["MSE", "Huber", "Huber", "Huber"] and list(cfg.weight_decay) == [0.0, 1e-6, 1e-6, 0.0]
              and cfg.numerical_masking.tolist()[1] == 0.05 and cfg.numerical_masking.tolist()[3] == 0.15
              and cfg.grouped_categorical_masking.tolist()[3] == 0.02, "selected_models_manifest.json")
dsd = cfg.set_index(["dataset", "model"]).dev_MCC_SD
L.check("S4.7.dev_sd_ratio_unsw", "Sec. 4.7", "development SD ratio AE/DAE, UNSW-NB15", dsd["UNSW-NB15", "AE"] / dsd["UNSW-NB15", "DAE"], "13.0", "confirmation_seed_results.csv")
L.check("S4.7.dev_sd_ratio_cic", "Sec. 4.7", "development SD ratio AE/DAE, CICIDS2017", dsd["CICIDS2017", "AE"] / dsd["CICIDS2017", "DAE"], "3.2", "confirmation_seed_results.csv")
write_csv(cfg, RESULTS / "tables" / "Table_3_5_selected_configurations.csv")
write_csv(pd.concat(rank_rows), RESULTS / "tables" / "Table_3_10_selection_ranking_reproduced.csv")

### 2. final-test levels and paired contrasts (Table 10; Fig. 8)

In [ ]:
levels, contrasts, tests, fig8 = [], [], [], []
for ds, name in DS.items():
    fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv"))
    pv = read_csv(art(ds, "statistical_analysis", "paired_seed_values.csv"))
    fb = read_csv(art(ds, "statistical_analysis", "paired_seed_bootstrap_effects.csv")).set_index("metric")
    L.assert_true(f"fin.{ds}.seeds", "Sec. 4.7", "five final seeds per model", all(
        sorted(fin[fin.model_type == m].seed) == list(FINAL_SEEDS) for m in ("AE", "DAE")), "final_seed_results.csv")
    a = fin[fin.model_type == "AE"].set_index("seed").sort_index()
    d = fin[fin.model_type == "DAE"].set_index("seed").sort_index()
    ok = all(np.allclose(pv[pv.metric == m].sort_values("seed")[["AE", "DAE"]].to_numpy(),
                         np.c_[a[m].to_numpy(), d[m].to_numpy()], rtol=1e-13, atol=0) for m in METRICS)
    L.assert_true(f"fin.{ds}.paired_values", "Table 10", "paired_seed_values.csv equals final_seed_results.csv (to 13 significant digits; the files store different digit counts)", ok, "both files")
    for m in METRICS + ["TP", "FP", "best_epoch"]:
        levels.append({"dataset": name, "metric": m, "AE_mean": a[m].mean(), "AE_sd": a[m].std(ddof=1),
                       "DAE_mean": d[m].mean(), "DAE_sd": d[m].std(ddof=1)})
    for m in METRICS:
        diff = (d[m] - a[m]).to_numpy()
        sm = summarise(diff)
        mc_lo, mc_hi = bootstrap_interval(diff)
        sign = "positive" if sm["mean_diff"] > 0 else "negative"
        contrasts.append({"dataset": name, "metric": m, "mean_DAE_minus_AE": sm["mean_diff"], "median": sm["median_diff"],
                          "sd_diff": sm["sd_diff"], "dz": sm["dz"], "consistency": f"{sm['n_same_sign']}/5 {sign}",
                          "exact_bootstrap_lo": sm["exact_lo"], "exact_bootstrap_hi": sm["exact_hi"],
                          "mc_bootstrap_lo_seed2026": mc_lo, "mc_bootstrap_hi_seed2026": mc_hi,
                          "frozen_NB08_lo": fb.loc[m, "CI_lower"], "frozen_NB08_hi": fb.loc[m, "CI_upper"],
                          "frozen_NB08_mean": fb.loc[m, "mean_difference"], "frozen_NB08_dz": fb.loc[m, "cohen_dz"]})
        tests.append({"dataset": name, "metric": m, **{k: sm[k] for k in ("t", "p_t", "t_lo", "t_hi", "p_sign", "p_wilcoxon")}})
        if m in ("MCC", "Recall", "PR-AUC", "FP_per_1000_benign"):
            fig8.append({"dataset": name, "metric": m, "mean": sm["mean_diff"], "lo": sm["exact_lo"], "hi": sm["exact_hi"]})
    n = int(a.iloc[0][["TP", "FP", "FN", "TN"]].sum())
    L.check(f"S4.7.{ds}.n", "Sec. 4.7", "final-test observations", n, {"cicids2017": "668,562", "unsw_nb15": "525,584"}[ds], "final_seed_results.csv")
    L.check(f"S4.7.{ds}.n_attack", "Sec. 4.7", "final-test attacks", int(a.iloc[0].TP + a.iloc[0].FN),
            {"cicids2017": "327,595", "unsw_nb15": "192,770"}[ds], "final_seed_results.csv")
lev = pd.DataFrame(levels).set_index(["dataset", "metric"])
con = pd.DataFrame(contrasts)
tst = pd.DataFrame(tests)

T10 = {  # dataset, metric: (mean, consistency, lo, hi, dz) as printed in Table 10
    ("CICIDS2017", "MCC"): ("-0.005508", "5/5 negative", "-0.008170", "-0.002845", "-1.569"),
    ("CICIDS2017", "Recall"): ("-0.006499", "5/5 negative", "-0.009926", "-0.003128", "-1.433"),
    ("CICIDS2017", "F1"): ("-0.006307", "5/5 negative", "-0.009535", "-0.003094", "-1.468"),
    ("CICIDS2017", "Precision"): ("-0.000824", "5/5 negative", "-0.001170", "-0.000491", "-1.861"),
    ("CICIDS2017", "Balanced_Accuracy"): ("-0.003359", "5/5 negative", "-0.005043", "-0.001676", "-1.499"),
    ("CICIDS2017", "PR-AUC"): ("-0.012958", "5/5 negative", "-0.018171", "-0.007694", "-2.003"),
    ("CICIDS2017", "FPR"): ("0.000220", "5/5 positive", "0.000069", "0.000371", "1.103"),
    ("CICIDS2017", "FP_per_1000_benign"): ("0.220", "5/5 positive", "0.069", "0.371", "1.103"),
    ("UNSW-NB15", "MCC"): ("0.003025", "5/5 positive", "0.001076", "0.005099", "1.177"),
    ("UNSW-NB15", "Recall"): ("0.004411", "5/5 positive", "0.001910", "0.006913", "1.377"),
    ("UNSW-NB15", "F1"): ("0.002219", "5/5 positive", "0.000821", "0.003645", "1.221"),
    ("UNSW-NB15", "Precision"): ("-0.000416", "3/5 negative", "-0.001263", "0.000368", "-0.408"),
    ("UNSW-NB15", "Balanced_Accuracy"): ("0.002069", "5/5 positive", "0.000814", "0.003324", "1.283"),
    ("UNSW-NB15", "PR-AUC"): ("-0.005571", "3/5 negative", "-0.012528", "0.000499", "-0.665"),
    ("UNSW-NB15", "FPR"): ("0.000273", "4/5 positive", "-0.000138", "0.000733", "0.496"),
    ("UNSW-NB15", "FP_per_1000_benign"): ("0.273", "4/5 positive", "-0.138", "0.733", "0.496")}
ci = con.set_index(["dataset", "metric"])
for (dsn, m), (mu, cons, lo, hi, dz) in T10.items():
    r = ci.loc[(dsn, m)]
    k = f"T10.{dsn}.{m}"
    L.check(k + ".mean", "Table 10", "mean DAE - AE", r.mean_DAE_minus_AE, mu, "paired_seed_values.csv")
    L.assert_true(k + ".consistency", "Table 10", f"seed-direction consistency {cons}", r.consistency == cons, "paired_seed_values.csv")
    L.check(k + ".lo", "Table 10", "95% interval lower bound (exact enumeration)", r.exact_bootstrap_lo, lo, "paired_seed_values.csv")
    L.check(k + ".hi", "Table 10", "95% interval upper bound (exact enumeration)", r.exact_bootstrap_hi, hi, "paired_seed_values.csv")
    L.check(k + ".dz", "Table 10", "d_z", r.dz, dz, "paired_seed_values.csv")
for (dsn, m), r in ci.iterrows():
    for side in ("lo", "hi"):
        if not np.isclose(r[f"exact_bootstrap_{side}"], r[f"frozen_NB08_{side}"], rtol=0, atol=5e-7 * max(1, abs(r[f"frozen_NB08_{side}"]))):
            L.note(f"NB08.{dsn}.{m}.{side}", "Table 10 notes 1-2", f"frozen NB-08 bootstrap {side} bound differs from exact enumeration",
                   f"exact {r[f'exact_bootstrap_{side}']:.6g} vs frozen {r[f'frozen_NB08_{side}']:.6g}", "paired_seed_bootstrap_effects.csv",
                   "FROZEN_TABLE_DISCREPANCY")

for (dsn, m), p in {("CICIDS2017", "MCC"): "-0.0055", ("CICIDS2017", "Recall"): "-0.0065", ("UNSW-NB15", "MCC"): "0.0030",
                    ("UNSW-NB15", "Recall"): "0.0044", ("CICIDS2017", "PR-AUC"): "-0.0130", ("UNSW-NB15", "PR-AUC"): "-0.0056",
                    ("CICIDS2017", "FP_per_1000_benign"): "0.220", ("UNSW-NB15", "FP_per_1000_benign"): "0.273"}.items():
    L.check(f"abstract.{dsn}.{m}", "Abstract", f"mean DAE - AE {m}", ci.loc[(dsn, m), "mean_DAE_minus_AE"], p, "paired_seed_values.csv")
L.assert_true("abstract.reversal", "Abstract; highlights", "MCC and recall contrasts have opposite signs in the two corpora, 5/5 seeds each",
              all(ci.loc[("CICIDS2017", m), "consistency"] == "5/5 negative" and ci.loc[("UNSW-NB15", m), "consistency"] == "5/5 positive"
                  for m in ("MCC", "Recall")), "paired_seed_values.csv")

# levels quoted in the text
for (dsn, m, who), p in {("CICIDS2017", "MCC", "AE_mean"): "0.522889", ("CICIDS2017", "MCC", "AE_sd"): "0.003799",
                         ("CICIDS2017", "MCC", "DAE_mean"): "0.517382", ("CICIDS2017", "MCC", "DAE_sd"): "0.001169",
                         ("CICIDS2017", "Recall", "AE_mean"): "0.445567", ("CICIDS2017", "Recall", "DAE_mean"): "0.439068",
                         ("CICIDS2017", "PR-AUC", "AE_mean"): "0.861488", ("CICIDS2017", "PR-AUC", "DAE_mean"): "0.848530",
                         ("UNSW-NB15", "MCC", "AE_mean"): "0.904585", ("UNSW-NB15", "MCC", "AE_sd"): "0.004759",
                         ("UNSW-NB15", "MCC", "DAE_mean"): "0.907611", ("UNSW-NB15", "MCC", "DAE_sd"): "0.004051",
                         ("UNSW-NB15", "Recall", "AE_mean"): "0.895536", ("UNSW-NB15", "Recall", "DAE_mean"): "0.899947",
                         ("UNSW-NB15", "PR-AUC", "AE_mean"): "0.981355", ("UNSW-NB15", "PR-AUC", "DAE_mean"): "0.975784",
                         ("UNSW-NB15", "best_epoch", "AE_mean"): "149.0", ("UNSW-NB15", "best_epoch", "AE_sd"): "1.22",
                         ("UNSW-NB15", "best_epoch", "DAE_mean"): "108.6"}.items():
    L.check(f"S4.7.level.{dsn}.{m}.{who}", "Sec. 4.7", f"{who.replace('_', ' ')} of {m}", lev.loc[(dsn, m), who], p, "final_seed_results.csv")
for dsn, p in (("UNSW-NB15", "1.17"), ("CICIDS2017", "3.25")):
    L.check(f"S4.7.final_sd_ratio.{dsn}", "Sec. 4.7", "final SD ratio AE/DAE", lev.loc[(dsn, "MCC"), "AE_sd"] / lev.loc[(dsn, "MCC"), "DAE_sd"], p, "final_seed_results.csv")
L.check("S4.7.unsw_mcc_in_ae_sd", "Sec. 4.7; Table 10", "UNSW-NB15 MCC contrast in AE seed SDs",
        ci.loc[("UNSW-NB15", "MCC"), "mean_DAE_minus_AE"] / lev.loc[("UNSW-NB15", "MCC"), "AE_sd"], "0.636", "final_seed_results.csv")
for dsn, tp, fp in (("CICIDS2017", "-2129", "75"), ("UNSW-NB15", "850", "91")):
    L.check(f"S4.7.tp_diff.{dsn}", "Sec. 4.7; Table 10", "mean TP difference per run (DAE - AE)",
            lev.loc[(dsn, "TP"), "DAE_mean"] - lev.loc[(dsn, "TP"), "AE_mean"], tp + ".", "final_seed_results.csv") if False else \
        L.check(f"S4.7.tp_diff.{dsn}", "Sec. 4.7; Table 10", "mean TP difference per run (DAE - AE), nearest integer",
                round(lev.loc[(dsn, "TP"), "DAE_mean"] - lev.loc[(dsn, "TP"), "AE_mean"]), tp, "final_seed_results.csv")
    L.check(f"S4.7.fp_diff.{dsn}", "Sec. 4.7; Table 10", "mean FP difference per run (DAE - AE), nearest integer",
            round(lev.loc[(dsn, "FP"), "DAE_mean"] - lev.loc[(dsn, "FP"), "AE_mean"]), fp, "final_seed_results.csv")
fu = read_csv(art("unsw_nb15", "final_evaluation", "final_seed_results.csv"))
dae_u = fu[fu.model_type == "DAE"]
L.check("S4.7.unsw_dae_epoch150", "Sec. 4.7", "UNSW-NB15 DAE runs that reached the 150-epoch ceiling", int((dae_u.best_epoch == 150).sum()), "1", "final_seed_results.csv")
a202 = fu[(fu.model_type == "AE") & (fu.seed == 202)].iloc[0]
d202 = fu[(fu.model_type == "DAE") & (fu.seed == 202)].iloc[0]
L.assert_true("S4.7.unsw_seed202", "Sec. 4.7", "UNSW-NB15 seed 202: DAE better on both axes (more TP, fewer FP)",
              d202.TP > a202.TP and d202.FP < a202.FP, "final_seed_results.csv")
tc = tst[tst.dataset == "CICIDS2017"].set_index("metric")
L.assert_true("S4.7.cic_exact_tests", "Sec. 4.7", "CICIDS2017: exact sign and Wilcoxon tests give p = 0.0625 for every metric",
              bool(np.allclose(tc.p_sign, 0.0625) and np.allclose(tc.p_wilcoxon, 0.0625)), "paired_seed_values.csv")
six = ["MCC", "Recall", "F1", "Precision", "Balanced_Accuracy", "PR-AUC"]
L.check("S4.7.cic_t_min", "Sec. 4.7", "smallest paired-t p over the six detection metrics", tc.loc[six, "p_t"].min(), "0.011", "paired_seed_values.csv")
L.check("S4.7.cic_t_max", "Sec. 4.7", "largest paired-t p over the six detection metrics", tc.loc[six, "p_t"].max(), "0.033", "paired_seed_values.csv")
L.check("S4.7.cic_fpr_p", "Sec. 4.7", "paired-t p, FPR", tc.loc["FPR", "p_t"], "0.0691", "paired_seed_values.csv")
L.check("S4.7.cic_fpr_tlo", "Sec. 4.7", "paired-t interval lower, FPR", tc.loc["FPR", "t_lo"], "-2.756e-5", "paired_seed_values.csv")
L.check("S4.7.cic_fpr_thi", "Sec. 4.7", "paired-t interval upper, FPR", tc.loc["FPR", "t_hi"], "4.675e-4", "paired_seed_values.csv")
t19 = read_csv(art("cicids2017", "statistical_analysis", "T19_statistical_comparison.csv")).set_index("metric")
both = [m for m in METRICS if m in t19.index]
L.assert_true("NB08.T19.tests", "Sec. 4.7", "recomputed t, p_t, t-interval, sign and Wilcoxon p equal frozen T19 (CICIDS2017)",
              all(np.allclose(tc.loc[both, c].to_numpy(), t19.loc[both, c].to_numpy(), rtol=1e-9, atol=1e-12)
                  for c in ("t", "p_t", "t_lo", "t_hi", "p_sign", "p_wilcoxon")), "T19_statistical_comparison.csv")
write_csv(con, RESULTS / "tables" / "Table_3_4_AE_DAE_contrasts.csv")
write_csv(lev.reset_index(), RESULTS / "tables" / "Table_3_6_final_test_levels.csv")
write_csv(tst, RESULTS / "tables" / "Table_3_7_statistical_tests.csv")
write_csv(pd.DataFrame(fig8), RESULTS / "figure_source_data" / "Figure_3_5_data.csv")

### 3. denominators, score geometry, families, development-versus-final (archive-derived tables)

In [ ]:
geo_rows, dvf_rows = [], []
for ds, name in DS.items():
    rs = read_csv(art(ds, "final_evaluation", "derived_from_archives", "run_summary_final_test.csv"))
    rd = read_csv(art(ds, "model_selection", "derived_from_archives", "run_summary_development.csv"))
    L.assert_true(f"arch.{ds}.integrity", "Supplementary S1", "final archives: one threshold, strict-exceedance rule, unique row_id, one prediction per group, no mixed groups",
                  bool(rs.prediction_rule_consistent.all() and (rs.threshold_unique_values == 1).all() and rs.row_id_unique.all()
                       and (rs.n_fingerprints_with_varying_prediction == 0).all() and (rs.n_mixed_label_fingerprints == 0).all()),
                  "run_summary_final_test.csv")
    L.assert_true(f"arch.{ds}.dev_integrity", "Supplementary S1", "development archives: same integrity properties",
                  bool(rd.prediction_rule_consistent.all() and rd.row_id_unique.all() and (rd.n_fingerprints_with_varying_prediction == 0).all()),
                  "run_summary_development.csv")
    fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv"))
    mm = rs.merge(fin, left_on=["config_hash", "seed"], right_on=["config_hash", "seed"], suffixes=("", "_frozen"))
    L.assert_true(f"arch.{ds}.final_match", "Supplementary S1", "archive-derived metrics equal final_seed_results.csv for all 10 runs",
                  len(mm) == 10 and all(np.allclose(mm[c], mm[c + "_frozen"], rtol=0, atol=1e-12) for c in
                                        ["MCC", "Recall", "F1", "Precision", "PR-AUC", "FPR", "Balanced_Accuracy", "ROC-AUC",
                                         "Average_Precision", "TPR_at_1pct_FPR", "TP", "FP", "FN", "TN", "threshold"])
                  and bool((mm.run_hash == mm.run_hash_frozen).all()), "run_summary_final_test.csv vs final_seed_results.csv")
    conf = read_csv(art(ds, "model_selection", "confirmation_seed_results.csv"))
    md = rd.merge(conf, on=["config_hash", "seed"], suffixes=("", "_frozen"))
    L.assert_true(f"arch.{ds}.dev_match", "Supplementary S1", "archive-derived development metrics equal confirmation_seed_results.csv",
                  len(md) == len(rd) and all(np.allclose(md[c], md[c + "_frozen"], rtol=0, atol=1e-12) for c in
                                             ["MCC", "Recall", "PR-AUC", "FPR", "TP", "FP", "FN", "TN", "threshold"]),
                  "run_summary_development.csv vs confirmation_seed_results.csv")
    n_rh = int((md.run_hash != md.run_hash_frozen).sum())
    if n_rh:
        L.note(f"arch.{ds}.dev_run_hash", "Supplementary S1", "development archives whose stored run_hash differs from the confirmation table (metrics identical)",
               f"{n_rh} of {len(md)}: " + ", ".join(f"{r.model} {r.config_hash[:8]} seed {r.seed}" for r in md[md.run_hash != md.run_hash_frozen].itertuples()),
               "run_summary_development.csv vs confirmation_seed_results.csv", "CORRECTION_NEEDED")
    for m in ("AE", "DAE"):
        r = rs[rs.model == m]
        geo_rows.append({"dataset": name, "model": m, "recall_per_row": r.Recall.mean(), "recall_per_attack_fingerprint": r.recall_per_attack_fingerprint.mean(),
                         "n_attack_fingerprints": int(r.n_attack_fingerprints.iloc[0]),
                         "missed_median_over_threshold": r.missed_attack_median_error_over_threshold.mean(),
                         "missed_p90_over_threshold": r.missed_attack_p90_error_over_threshold.mean(),
                         "attack_below_benign_median_pct": 100 * r.attack_share_below_benign_median.mean(),
                         "attack_at_or_below_threshold_pct": 100 * r.attack_share_at_or_below_threshold.mean()})
    # per-fingerprint recall contrast
    ra = rs[rs.model == "AE"].set_index("seed").sort_index()
    rdd = rs[rs.model == "DAE"].set_index("seed").sort_index()
    dfp = (rdd.recall_per_attack_fingerprint - ra.recall_per_attack_fingerprint).to_numpy()
    drow = (rdd.Recall - ra.Recall).to_numpy()
    lo, hi = exact_interval(dfp)
    geo_rows[-1].update({"per_vector_contrast": dfp.mean(), "per_vector_lo": lo, "per_vector_hi": hi,
                         "per_vector_same_sign_as_row_all_seeds": bool((np.sign(dfp) == np.sign(drow)).all())})
    # development versus final, like-for-like
    finc = fin[fin.seed.isin(CONFIRMATION_SEEDS)]
    sel = load_json(art(ds, "model_selection", "selected_models_manifest.json"))
    for m in ("AE", "DAE"):
        h = sel[f"selected_{m}"]["config_hash"]
        dv = rd[(rd.config_hash == h) & rd.seed.isin(CONFIRMATION_SEEDS)]
        fm = finc[finc.model_type == m].MCC.mean()
        dvf_rows.append({"dataset": name, "model": m, "dev_MCC": dv.MCC.mean(), "dev_MCC_excluding_mixed": dv.MCC_excluding_mixed_label_groups.mean(),
                         "final_MCC_confirmation_seeds": fm, "final_minus_dev": fm - dv.MCC.mean(),
                         "final_minus_dev_excluding_mixed": fm - dv.MCC_excluding_mixed_label_groups.mean(),
                         "dev_mixed_rows": int(dv.mixed_label_rows.iloc[0]), "dev_mixed_attack_rows": int(dv.mixed_label_attack_rows.iloc[0]),
                         "dev_mixed_attack_rows_detected_max": int(dv.mixed_label_attack_rows_detected.max())})
geo = pd.DataFrame(geo_rows).set_index(["dataset", "model"])
dvf = pd.DataFrame(dvf_rows).set_index(["dataset", "model"])
for (dsn, m), p in {("UNSW-NB15", "AE"): "0.6285", ("UNSW-NB15", "DAE"): "0.6451", ("CICIDS2017", "AE"): "0.5876", ("CICIDS2017", "DAE"): "0.5792"}.items():
    L.check(f"S4.7.fp_recall.{dsn}.{m}", "Sec. 4.7", "recall per distinct attack fingerprint (5-seed mean)", geo.loc[(dsn, m), "recall_per_attack_fingerprint"], p, "run_summary_final_test.csv")
for dsn, p in (("UNSW-NB15", "46,798"), ("CICIDS2017", "248,115")):
    L.check(f"S4.7.n_attack_fp.{dsn}", "Sec. 4.7", "distinct attack vectors in the final test", geo.loc[(dsn, "AE"), "n_attack_fingerprints"], p, "run_summary_final_test.csv")
for dsn, (mu, lo, hi) in {"UNSW-NB15": ("0.0166", "0.0061", "0.0270"), "CICIDS2017": ("-0.0084", "-0.0126", "-0.0042")}.items():
    r = geo.loc[(dsn, "DAE")]
    L.check(f"S4.7.fp_contrast.{dsn}", "Sec. 4.7", "per-vector recall contrast DAE - AE", r.per_vector_contrast, mu, "run_summary_final_test.csv")
    L.check(f"S4.7.fp_contrast_lo.{dsn}", "Sec. 4.7", "per-vector contrast interval lower", r.per_vector_lo, lo, "run_summary_final_test.csv")
    L.check(f"S4.7.fp_contrast_hi.{dsn}", "Sec. 4.7", "per-vector contrast interval upper", r.per_vector_hi, hi, "run_summary_final_test.csv")
    L.assert_true(f"S4.7.fp_contrast_sign.{dsn}", "Sec. 4.7", "per-vector and per-row contrasts share sign at all five seeds", r.per_vector_same_sign_as_row_all_seeds, "run_summary_final_test.csv")
L.check("S4.7.fp_contrast_ratio", "Sec. 4.7", "UNSW-NB15 per-vector contrast / per-row contrast",
        geo.loc[("UNSW-NB15", "DAE"), "per_vector_contrast"] / ci.loc[("UNSW-NB15", "Recall"), "mean_DAE_minus_AE"], "3.8", "as above")
GEO = {("CICIDS2017", "AE"): ("0.0332", "0.2344", "7.146", "55.44"), ("UNSW-NB15", "AE"): ("0.1767", "0.6720", "0.000", "10.45"),
       ("CICIDS2017", "DAE"): ("0.0294", "0.2202", "10.92", "56.09"), ("UNSW-NB15", "DAE"): ("0.2183", "0.7556", None, "10.01")}
for (dsn, m), (med, p90, bm, bt) in GEO.items():
    r = geo.loc[(dsn, m)]
    L.check(f"S4.7.geo.{dsn}.{m}.median", "Sec. 4.7", "median error of missed attacks / threshold", r.missed_median_over_threshold, med, "run_summary_final_test.csv")
    L.check(f"S4.7.geo.{dsn}.{m}.p90", "Sec. 4.7", "90th percentile error of missed attacks / threshold", r.missed_p90_over_threshold, p90, "run_summary_final_test.csv")
    if bm is not None:
        L.check(f"S4.7.geo.{dsn}.{m}.below_median", "Sec. 4.7", "attacks below the benign median (%)", r.attack_below_benign_median_pct, bm, "run_summary_final_test.csv")
    L.check(f"S4.7.geo.{dsn}.{m}.below_thr", "Sec. 4.7", "attacks at or below the threshold (%)", r.attack_at_or_below_threshold_pct, bt, "run_summary_final_test.csv")
L.assert_true("S4.7.geo.unsw_dae_below_median", "Sec. 4.7", "UNSW-NB15 DAE: fewer than 0.01% of attacks below the benign median",
              geo.loc[("UNSW-NB15", "DAE"), "attack_below_benign_median_pct"] < 0.01, "run_summary_final_test.csv")
DVF = {("CICIDS2017", "AE"): ("-0.006471", "-0.0193", "0.530234", "0.543028"), ("CICIDS2017", "DAE"): ("-0.007037", "-0.0197", "0.524699", "0.537336"),
       ("UNSW-NB15", "AE"): ("0.004026", "-0.0008", "0.903582", "0.908416"), ("UNSW-NB15", "DAE"): ("0.004073", "-0.0007", "0.906182", "0.911003")}
for (dsn, m), (raw, ex, dmcc, dex) in DVF.items():
    r = dvf.loc[(dsn, m)]
    if (dsn, m) == ("UNSW-NB15", "DAE"):   # printed as +0.004073; the value is 0.0040736, i.e. +0.004074 when rounded
        L.note("S4.7.dvf.UNSW-NB15.DAE.raw", "Sec. 4.7", "final minus development MCC, UNSW-NB15 DAE: manuscript prints +0.004073 (truncated)",
               f"{r.final_minus_dev:.7f} (rounds to +0.004074)", "run summaries", "CORRECTION_NEEDED")
        L.check(f"S4.7.dvf.{dsn}.{m}.raw4", "Sec. 4.7", "final minus development MCC (confirmation seeds), 4 d.p.", r.final_minus_dev, "0.0041", "run summaries")
    else:
        L.check(f"S4.7.dvf.{dsn}.{m}.raw", "Sec. 4.7", "final minus development MCC (confirmation seeds)", r.final_minus_dev, raw, "run summaries")
    L.check(f"S4.7.dvf.{dsn}.{m}.excl", "Sec. 4.7", "final minus development MCC, mixed-label groups excluded from development", r.final_minus_dev_excluding_mixed, ex, "run summaries")
    L.check(f"S4.7.dvf.{dsn}.{m}.dev", "Sec. 4.7", "development MCC", r.dev_MCC, dmcc, "run_summary_development.csv")
    L.check(f"S4.7.dvf.{dsn}.{m}.dev_excl", "Sec. 4.7", "development MCC excluding mixed-label groups", r.dev_MCC_excluding_mixed, dex, "run_summary_development.csv")
L.check("S4.7.cic_mixed_rows", "Sec. 4.7", "CICIDS2017 mixed-label observations in development", dvf.loc[("CICIDS2017", "AE"), "dev_mixed_rows"], "7,020", "run_summary_development.csv")
L.assert_true("S4.7.cic_mixed_missed", "Sec. 4.7", "all 6,284 attack rows in CICIDS2017 mixed groups missed by both models at every confirmation seed",
              dvf.loc["CICIDS2017"].dev_mixed_attack_rows.eq(6284).all() and dvf.loc["CICIDS2017"].dev_mixed_attack_rows_detected_max.eq(0).all(),
              "run_summary_development.csv")

fam_all = []
for ds, name in DS.items():
    ft = family_table(read_csv(art(ds, "final_evaluation", "derived_from_archives", "family_detection_final_test.csv")))
    fam_all.append(ft.assign(dataset=name))
fam = pd.concat(fam_all).set_index(["dataset", "attack_family"])
S3 = {("CICIDS2017", "DoS Hulk"): (135746, 99570, "0.658", "0.655", "0.897", "0.893", "25.6"),
      ("CICIDS2017", "PortScan"): (93365, 52762, "0.007", "0.004", "0.010", "0.008", "51.1"),
      ("CICIDS2017", "DDoS"): (75211, 75204, "0.636", "0.632", "0.636", "0.632", "15.1"),
      ("CICIDS2017", "DoS Slowhttptest"): (3231, 3142, "0.752", "0.438", "0.745", "0.423", "0.4"),
      ("UNSW-NB15", "Generic"): (129617, 11303, "0.994", "0.994", "0.932", "0.935", "3.9"),
      ("UNSW-NB15", "Exploits"): (26817, 16528, "0.880", "0.891", "0.810", "0.827", "15.9"),
      ("UNSW-NB15", "Fuzzers"): (14005, 12255, "0.359", "0.371", "0.312", "0.325", "44.6"),
      ("UNSW-NB15", "Reconnaissance"): (8473, 6079, "0.382", "0.412", "0.463", "0.494", "26.0"),
      ("UNSW-NB15", "Worms"): (95, 93, "0.836", "0.886", "0.832", "0.884", "0.1")}
for (dsn, f), (n, v, ar, dr, av, dv_, ms) in S3.items():
    r = fam.loc[(dsn, f)]
    k = f"S3.{dsn}.{f}"
    L.check(k + ".rows", "Table S3", "rows", r.rows, str(n), "family_detection_final_test.csv")
    L.check(k + ".vectors", "Table S3", "distinct vectors", r.distinct_vectors, str(v), "family_detection_final_test.csv")
    L.check(k + ".AE_rec", "Table S3", "AE recall per row", r.AE_recall_per_row, ar, "family_detection_final_test.csv")
    L.check(k + ".DAE_rec", "Table S3", "DAE recall per row", r.DAE_recall_per_row, dr, "family_detection_final_test.csv")
    L.check(k + ".AE_recv", "Table S3", "AE recall per vector", r.AE_recall_per_vector, av, "family_detection_final_test.csv")
    L.check(k + ".DAE_recv", "Table S3", "DAE recall per vector", r.DAE_recall_per_vector, dv_, "family_detection_final_test.csv")
    L.check(k + ".miss", "Table S3", "share of AE misses (%)", r.AE_miss_share_pct, ms, "family_detection_final_test.csv")
gen = fam.loc[("UNSW-NB15", "Generic")]
tot_u = fam.loc["UNSW-NB15"]
L.check("S4.7.generic_rows_pct", "Sec. 4.7", "Generic share of UNSW-NB15 attack rows (%)", 100 * gen.rows / tot_u.rows.sum(), "67.2", "family_detection_final_test.csv")
L.check("S4.7.generic_vec_pct", "Sec. 4.7", "Generic share of the 46,798 distinct attack vectors (%)",
        100 * gen.distinct_vectors / geo.loc[("UNSW-NB15", "AE"), "n_attack_fingerprints"], "24.2", "family_detection_final_test.csv + run_summary_final_test.csv")
ru = read_csv(art("unsw_nb15", "final_evaluation", "derived_from_archives", "run_summary_final_test.csv")).iloc[0]
L.note("S3.unsw_multi_family", "Table S3", "UNSW-NB15 attack fingerprints labelled with more than one attack family (family 'Vectors' therefore overlap)",
       f"{int(ru.n_attack_fingerprints_with_several_families)} fingerprints, {int(ru.attack_rows_in_several_family_fingerprints)} rows; "
       f"sum of per-family vectors {int(tot_u.distinct_vectors.sum())} vs {int(ru.n_attack_fingerprints)} distinct attack vectors",
       "run_summary_final_test.csv + family_detection_final_test.csv", "CORRECTION_NEEDED")
L.assert_true("S3.cic_families", "Table S3", "14 CICIDS2017 and 9 UNSW-NB15 attack families in the final test",
              len(fam.loc["CICIDS2017"]) == 14 and len(tot_u) == 9, "family_detection_final_test.csv")
write_csv(geo.reset_index(), RESULTS / "tables" / "Table_3_8_denominator_and_score_geometry.csv")
write_csv(dvf.reset_index(), RESULTS / "tables" / "Table_3_9_development_vs_final.csv")
write_csv(fam.reset_index(), RESULTS / "tables" / "Table_S3_family_recall.csv")
L.note("S4.7.matched_ablation", "Sec. 4.7", "matched-ablation contrasts (-0.004998, +0.001036, d_z -0.155; +0.010377, d_z 1.755)",
       "not recomputable: NB-04 ablation outputs are not in the evidence set", "analysis record", "NOT_REPRODUCIBLE_HERE")
assert L.finish() == 0, "some checks FAILED - see the claim log below"

## Results

In [ ]:
# ===== Results of the checks =====
import pandas as pd
from IPython.display import display
log = pd.read_csv(REPO_ROOT / "results" / "claims_check" / "reproduce_final_statistics.csv")
print(log.status.value_counts().to_string())
display(log[log.status != "PASS"])          # notes: values that cannot be recomputed here, or corrections
display(log.head(15))                        # first passing checks (all rows are in the CSV)

In [ ]:
# ===== Tables written by this notebook =====
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', 30)
print("results/tables/Table_3_5_selected_configurations.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_5_selected_configurations.csv"))
print("results/tables/Table_3_4_AE_DAE_contrasts.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_4_AE_DAE_contrasts.csv"))
print("results/tables/Table_3_8_denominator_and_score_geometry.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_8_denominator_and_score_geometry.csv"))
print("results/tables/Table_3_9_development_vs_final.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_9_development_vs_final.csv"))
print("results/tables/Table_S3_family_recall.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_S3_family_recall.csv"))